<div style = "background-color:#F5F7FB; text-align:left; border-left:5px solid #2E86DE; padding:2px; margin:0;">
    <h1 style = "color:#1F4E79; font-family:calibri; margin:0;">Discount Strategy What-If Scenario Dashboard</h1>
    <p style = "color:orange; font-family:calibri; margin:0;">Interactive Pricing Simulation for Revenue & Profit Analysis</p>
</div>

<div style="background-color:#F5F7FB; border-top:3px solid orange; color:#1F4E79; padding:5px;">
<h3>1. Creating a Safe SQLAlchemy engine for PostgreSQL DataBase</h3>
</div>


In [19]:
# Importing Operating System & SQLAlchemy Packages
import os
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

# IMPORTANT: for security, I use environment variables
db_user = os.getenv("PGUSER", "postgres")
db_password = os.getenv("PGPASSWORD", "PAM@Postgres")
db_host = os.getenv("PGHOST", "localhost")
db_port = os.getenv("PGPORT", "5432")
db_name = os.getenv("PGDB", "Scenario_DB")

connection_url = URL.create(
        "postgresql+psycopg2",
        username = db_user,
        password = db_password,
        host = db_host,
        port = db_port,
        database = db_name
)

engine = create_engine(connection_url)
print("Engine created sucessfully✅. Test connection below...")

# Quick Test
with engine.connect() as conn:
    r = conn.execute(text("SELECT 1")).scalar()
    print("Test query returned:", r)

Engine created sucessfully✅. Test connection below...
Test query returned: 1


## <span style="background-color:#F5F7FB; color:#1F4E79; border-radius:8px; padding:10px;">2. Uploading CSV file into DataFrame</span>

In [2]:
import pandas as pd
# Load the uploaded CSV file
df = pd.read_csv("super_store_flat.csv", encoding = "latin1")

# Confirm dataset loaded
print("Dataset loaded successfully✅.")
print("Shape of dataset:", df.shape)

# Preview first 5 rows
df.head()

Dataset loaded successfully✅.
Shape of dataset: (9994, 21)


,ï»¿row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,ountry,city,...,postal_code,region,product_id,category,sub_category,product _name,sales,quantity,discount,profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


## <span style="background-color:#F5F7FB; color:#1F4E79; border-radius:8px; padding:10px;">3. Checking the Table Structure in the DataFrame & Missing Values</span>

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   ï»¿row_id      9994 non-null   int64  
 1   order_id       9994 non-null   object 
 2   order_date     9994 non-null   object 
 3   ship_date      9994 non-null   object 
 4   ship_mode      9994 non-null   object 
 5   customer_id    9994 non-null   object 
 6   customer_name  9994 non-null   object 
 7   segment        9994 non-null   object 
 8   ountry         9994 non-null   object 
 9   city           9994 non-null   object 
 10  state          9994 non-null   object 
 11  postal_code    9994 non-null   int64  
 12  region         9994 non-null   object 
 13  product_id     9994 non-null   object 
 14  category       9994 non-null   object 
 15  sub_category   9994 non-null   object 
 16  product _name  9994 non-null   object 
 17  sales          9994 non-null   float64
 18  quantity

In [4]:
df.isnull().sum()

ï»¿row_id        0
order_id         0
order_date       0
ship_date        0
ship_mode        0
customer_id      0
customer_name    0
segment          0
ountry           0
city             0
state            0
postal_code      0
region           0
product_id       0
category         0
sub_category     0
product _name    0
sales            0
quantity         0
discount         0
profit           0
dtype: int64

## <span style="background-color:#F5F7FB; color:#1F4E79; border-radius:8px; padding:10px;">4. Fixing Data Types for Database Readiness</span>

In [5]:
# Convert Date Colums
df['order_date'] = pd.to_datetime(df['order_date'], errors = 'coerce')
df['ship_date'] = pd.to_datetime(df['ship_date'], errors = 'coerce')

# Convert Postal_code to string (safe for database storage)
df['postal_code'] = df['postal_code'].astype(str)

print("Data type corrections completed ✔")
print("\nUpdated Data Types:")
print(df.dtypes)

Data type corrections completed ✔

Updated Data Types:
ï»¿row_id                 int64
order_id                 object
order_date       datetime64[ns]
ship_date        datetime64[ns]
ship_mode                object
customer_id              object
customer_name            object
segment                  object
ountry                   object
city                     object
state                    object
postal_code              object
region                   object
product_id               object
category                 object
sub_category             object
product _name            object
sales                   float64
quantity                  int64
discount                float64
profit                  float64
dtype: object


## <span style="background-color:#F5F7FB; color:#1F4E79; border-radius:8px; padding:10px;">5. Creating What-If Table</span>

In [6]:
# Create Scenario Table
scenario_table = df[['order_date', 'sales', 'profit', 'discount','category', 'sub_category', 'region', 'segment']].drop_duplicates().reset_index(drop=True)

print("Scenario Table Created✔.")
print("Scenario Table:", scenario_table.shape)
scenario_table.head()


Scenario Table Created✔.
Scenario Table: (9992, 8)


,order_date,sales,profit,discount,category,sub_category,region,segment
0,2016-11-08,261.9600,41.9136,0.00,Furniture,Bookcases,South,Consumer
1,2016-11-08,731.9400,219.5820,0.00,Furniture,Chairs,South,Consumer
2,2016-06-12,14.6200,6.8714,0.00,Office Supplies,Labels,West,Corporate
3,2015-10-11,957.5775,-383.0310,0.45,Furniture,Tables,South,Consumer
4,2015-10-11,22.3680,2.5164,0.20,Office Supplies,Storage,South,Consumer


## <span style="background-color:#F5F7FB; color:#1F4E79; border-radius:8px; padding:10px;">6. Creating the SQL Database Table with Python</span>

In [7]:
with engine.connect() as conn:
    # Drop table if already exists (For Clean rebuild)
    conn.execute(text("DROP TABLE IF EXISTS scenario_table"))

# Create table DDL (run once). 
ddl_statements = [
    """
    CREATE TABLE IF NOT EXISTS scenario_table (
    order_date DATE,
    category VARCHAR (50),
    sub_category VARCHAR (100),
    region VARCHAR (50),
    segment VARCHAR (50),
    sales NUMERIC,
    profit NUMERIC,
    discount NUMERIC
    );
    """
]
with engine.begin() as conn:
    for s in ddl_statements:
        conn.exec_driver_sql(s)
print("DDL executed (Table created).✅")

DDL executed (Table created).✅


## <span style="background-color:#F5F7FB; color:#1F4E79; border-radius:8px; padding:10px;">7. Uploading Scenario Table into PostgreSQL</span>

In [8]:
import pandas as pd
scenario_table.to_sql('scenario_table', engine, if_exists = 'replace', index=False)
print("Scenario Table Uploaded Successfully")
scenario_table.head(8)

Scenario Table Uploaded Successfully


,order_date,sales,profit,discount,category,sub_category,region,segment
0,2016-11-08,261.9600,41.9136,0.00,Furniture,Bookcases,South,Consumer
1,2016-11-08,731.9400,219.5820,0.00,Furniture,Chairs,South,Consumer
2,2016-06-12,14.6200,6.8714,0.00,Office Supplies,Labels,West,Corporate
3,2015-10-11,957.5775,-383.0310,0.45,Furniture,Tables,South,Consumer
4,2015-10-11,22.3680,2.5164,0.20,Office Supplies,Storage,South,Consumer
5,2014-06-09,48.8600,14.1694,0.00,Furniture,Furnishings,West,Consumer
6,2014-06-09,7.2800,1.9656,0.00,Office Supplies,Art,West,Consumer
7,2014-06-09,907.1520,90.7152,0.20,Technology,Phones,West,Consumer


<div style = "background-color:#F5F7FB; padding:20px; border-top:3px solid orange;">
    <p style = "color:#1F4E79; font-size:14px; margin:0;">Created by <b>Pam S. George</b> | </p>
     <p style = "color:#1F4E79; font-style:italic; text-alignment:right;">Build on Resilience. Powered by Data | Confidential Reporting</p>
</div>